In [ ]:
import xgboost
import uproot
import apd
import ROOT
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mplhep
import os
from BDT_helper import add_columns, features, needed_cols, define_constants
ROOT.EnableImplicitMT()        # use all available cores

In [2]:
import sys
print(sys.executable)

/cvmfs/sft.cern.ch/lcg/views/LCG_109_swan/x86_64-el9-gcc13-opt/bin/python


In [3]:
# # Load trained model
# if 'eos' in os.getcwd():
#    File = "xgb_DD.root"
# else:
#     File = "root://eosuser.cern.ch//eos/user/k/ksowa/Analysis/xgb_DD.root"
channel = "BdToDmKstarp"
MODE = "DD"

constants = define_constants(channel)

file_odd = f"xgb_{MODE}_odd.root"
file_even = f"xgb_{MODE}_even.root"
bdt_odd = ROOT.TMVA.Experimental.RBDT(f"XGB_BDT_{MODE}_odd", file_odd)
bdt_even = ROOT.TMVA.Experimental.RBDT(f"XGB_BDT_{MODE}_even", file_even)

In [4]:
from apd import AnalysisData

datasets = AnalysisData("b2oc", "bd2dmkspi")
samples = datasets(eventtype="90000000", datatype="2025", filetype="data.root", name =  ["bdtodmkspi_sprucing25c1_down",
                                                                                         "bdtodmkspi_sprucing25c1_up",
                                                                                         "bdtodmkspi_sprucing25c2_down",
                                                                                         "bdtodmkspi_sprucing25c3_up"])

In [5]:
chain = ROOT.TChain("DecayTree")
for f in samples:
    file = ROOT.TFile.Open(f)
    for key in file.GetListOfKeys():
        if key.GetClassName() == "TDirectoryFile":
            dirname = key.GetName()
            if f"{MODE}_Kpipi" in dirname:
                chain.Add(f"{f}/{dirname}/DecayTree")
chain.SetBranchStatus("*", 0)
for f in needed_cols:
    chain.SetBranchStatus(f, 1)
rdf = ROOT.RDataFrame(chain)
rdf = add_columns(rdf)

In [6]:
particle = "KS0"
variable = "M"

limits = [460.0,540.0]
nbins = 80
unit = (limits[1]-limits[0])/nbins
m_ks0_pi = ROOT.RooRealVar("m_ks0_pi","m_ks0_pi",limits[0],limits[1])

canvas = ROOT.TCanvas("canvas","canvas",800,600)
hist_kstar = rdf.Histo1D((f"{particle}_{variable}", f"{particle}_{variable}; {variable} [MeV/c^2]; Entries", nbins, limits[0], limits[1]), f"{particle}_{variable}")
m_ks0_pi_hist = ROOT.RooDataHist("m_ks0_pi_hist","m_ks0_pi_hist",ROOT.RooArgList(m_ks0_pi),hist_kstar.GetValue())

#Create a drawable object and fill it
m_ks0_pi_plot = m_ks0_pi.frame(nbins) #Number of bins as an argument
m_ks0_pi_hist.plotOn(m_ks0_pi_plot, Name = "data")

#m_ks0_pi_plot.SetAxisRange(70.,751.)
m_ks0_pi_plot.SetTitle("")
ROOT.gPad.SetLeftMargin(0.15)
ROOT.gPad.SetBottomMargin(0.12)
m_ks0_pi_plot.GetYaxis().SetTitle("Candidates /(" + f'{unit:.2f}' + " MeV/#it{c}^{2})")
m_ks0_pi_plot.GetYaxis().SetTitleOffset(1.3)
m_ks0_pi_plot.GetXaxis().SetTitleOffset(1.0)
m_ks0_pi_plot.GetXaxis().SetTitleSize(0.05);
m_ks0_pi_plot.GetYaxis().SetTitleSize(0.05);
m_ks0_pi_plot.GetXaxis().SetTitle("#it{m}(#it{#pi^{#pm}#pi^{#mp}}) [MeV/#it{c}^{2}]")
m_ks0_pi_plot.Draw()
canvas.Draw()
#canvas.SetLogx(1)
#canvas.SaveAs("exercise_0.png")

In [7]:
canvas.SaveAs(f"m_pi_pi_{MODE}.pdf")

Info in <TCanvas::Print>: pdf file m_pi_pi_DD.pdf has been created


## Preselection

In [6]:
# RICH
rich_conditions = [
    "H1_PPHASRICH == 1",
    "H2_PPHASRICH == 1",
    "H3_PPHASRICH == 1",
    "Pi_Bachelor_PPHASRICH == 1",
    "Pi_1_KS0_PPHASRICH == 1",
    "Pi_2_KS0_PPHASRICH == 1",
]

for cond in rich_conditions:
    rdf = rdf.Filter(cond)

# MASS
mass_conditions = [
    f"abs(D_M - {constants['D_MASS']}) < 25",
    "B_M_K0Pi > 750",
    "B_M_K0Pi < 1150",
    #f"abs(B_M_K0Pi - {constants['KSTAR_MASS']}) < 75",

]
if MODE == "DD":
    mass_conditions += [f"abs(KS0_M - {constants['KS0_MASS']}) < 20"]
else:
    mass_conditions += [f"abs(KS0_M - {constants['KS0_MASS']}) < 15"]

for cond in mass_conditions:
    rdf = rdf.Filter(cond)

## BDT

In [7]:
ROOT.gInterpreter.Declare(rf"""
#ifndef BDT_EVAL_HELPER
#define BDT_EVAL_HELPER
#include <vector>
#include <ROOT/RVec.hxx>
#include "TMVA/RBDT.hxx"

using namespace TMVA::Experimental;

static RBDT odd_bdt("XGB_BDT_{MODE}_odd", "{file_odd}");
static RBDT even_bdt("XGB_BDT_{MODE}_even", "{file_even}");

float eval_odd_bdt(const ROOT::VecOps::RVec<float>& v) {{
    return odd_bdt.Compute(std::vector<float>(v.begin(), v.end()))[0];
}}

float eval_even_bdt(const ROOT::VecOps::RVec<float>& v) {{
    return even_bdt.Compute(std::vector<float>(v.begin(), v.end()))[0];
}}

// Cross-evaluation: odd events -> even BDT, even events -> odd BDT
float eval_bdt(ULong64_t eventNumber, const ROOT::VecOps::RVec<float>& v) {{
    std::vector<float> vf(v.begin(), v.end());
    if (eventNumber % 2 == 1)
        return even_bdt.Compute(vf)[0];
    else
        return odd_bdt.Compute(vf)[0];
}}
#endif
""")


True

In [8]:
features_str = ",".join(features)
expr_bdt = f"eval_bdt(EVENTNUMBER, ROOT::VecOps::RVec<double>{{{features_str}}})"
rdf = rdf.Define("bdt_score", expr_bdt)


In [9]:
# particle = "bdt"
# variable = "score"

# nbins = 100
# limits = [0,1]
# unit = (limits[1]-limits[0])/nbins


# can = ROOT.TCanvas()
# can.SetLeftMargin(0.12)
# hist = rdf.Histo1D((f"{particle}_{variable}", f"{particle}_{variable}; BDT score [-];" +"Entries", nbins, limits[0], limits[1]), f"{particle}_{variable}")
# hist.SetLineWidth(2)
# ROOT.gPad.SetLogy()
# hist.Draw()
# # ROOT.gPad.Update()  # forces ROOT to compute axis limits
# # ymax = ROOT.gPad.GetUymax()   # top of y axis
# # ymin = ROOT.gPad.GetUymin()   # bottom of y axis
# can.Draw()

In [ ]:
# particle = "B"
# variable = "M"

# nbins = 100
# limits = [5000,5700]
# unit = (limits[1]-limits[0])/nbins


# can = ROOT.TCanvas()
# can.SetLeftMargin(0.12)
# hist = rdf.Filter("bdt_score > 0.9").Histo1D((f"{particle}_{variable}", f"{particle}_{variable}; {variable} [MeV/c^2]; Entries", nbins, limits[0], limits[1]), f"{particle}_{variable}")
# hist.SetLineWidth(2)
# # ROOT.gPad.SetLogy()
# hist.Draw()
# # ROOT.gPad.Update()  # forces ROOT to compute axis limits
# # ymax = ROOT.gPad.GetUymax()   # top of y axis
# # ymin = ROOT.gPad.GetUymin()   # bottom of y axis
# can.Draw()

In [ ]:
particle = "B"
variable = "M"

limits = [5050.0,5650.0]
nbins = 60
unit = (limits[1]-limits[0])/nbins
m_b = ROOT.RooRealVar("m_b","m_b",limits[0],limits[1])
hist_b = rdf.Filter("bdt_score > 0.9").Histo1D((f"{particle}_{variable}", f"{particle}_{variable}; {variable} [MeV/c^2]; Entries", nbins, limits[0], limits[1]), f"{particle}_{variable}")
m_b_hist = ROOT.RooDataHist("m_b_hist","m_b_hist",ROOT.RooArgList(m_b),hist_b.GetValue())

#Plot the result.
#First create a canvas
canvas = ROOT.TCanvas("canvas","canvas",800,600)

#Create a drawable object and fill it
m_b_plot = m_b.frame(nbins) #Number of bins as an argument
m_b_hist.plotOn(m_b_plot, Name = "data")

#m_b_plot.SetAxisRange(70.,751.)
m_b_plot.SetTitle("")
ROOT.gPad.SetLeftMargin(0.15)
ROOT.gPad.SetBottomMargin(0.12)
m_b_plot.GetYaxis().SetTitle("Candidates /(" + f'{unit:.2f}' + " MeV/#it{c}^{2})")
m_b_plot.GetYaxis().SetTitleOffset(1.3)
m_b_plot.GetXaxis().SetTitleOffset(1.0)
m_b_plot.GetXaxis().SetTitleSize(0.05);
m_b_plot.GetYaxis().SetTitleSize(0.05);
m_b_plot.GetXaxis().SetTitle("#it{m}(#it{D^{#mp}K_{S}^{0}#pi^{#pm})} [MeV/#it{c}^{2}]")
m_b_plot.Draw()
canvas.Draw()
#canvas.SaveAs("exercise_0.png")

In [ ]:
particle = "B"
variable = "M_K0Pi"

limits = [750.0,1150.0]
nbins = 80
unit = (limits[1]-limits[0])/nbins
m_ks0_pi = ROOT.RooRealVar("m_ks0_pi","m_ks0_pi",limits[0],limits[1])

canvas = ROOT.TCanvas("canvas","canvas",800,600)
hist_kstar = rdf.Filter("bdt_score > 0.9").Histo1D((f"{particle}_{variable}", f"{particle}_{variable}; {variable} [MeV/c^2]; Entries", nbins, limits[0], limits[1]), f"{particle}_{variable}")
m_ks0_pi_hist = ROOT.RooDataHist("m_ks0_pi_hist","m_ks0_pi_hist",ROOT.RooArgList(m_ks0_pi),hist_kstar.GetValue())

#Create a drawable object and fill it
m_ks0_pi_plot = m_ks0_pi.frame(nbins) #Number of bins as an argument
m_ks0_pi_hist.plotOn(m_ks0_pi_plot, Name = "data")

#m_ks0_pi_plot.SetAxisRange(70.,751.)
m_ks0_pi_plot.SetTitle("")
ROOT.gPad.SetLeftMargin(0.15)
ROOT.gPad.SetBottomMargin(0.12)
m_ks0_pi_plot.GetYaxis().SetTitle("Candidates /(" + f'{unit:.2f}' + " MeV/#it{c}^{2})")
m_ks0_pi_plot.GetYaxis().SetTitleOffset(1.3)
m_ks0_pi_plot.GetXaxis().SetTitleOffset(1.0)
m_ks0_pi_plot.GetXaxis().SetTitleSize(0.05);
m_ks0_pi_plot.GetYaxis().SetTitleSize(0.05);
m_ks0_pi_plot.GetXaxis().SetTitle("#it{m}(#it{K_{S}^{0}#pi^{#pm})} [MeV/#it{c}^{2}]")
m_ks0_pi_plot.Draw()
canvas.Draw()
#canvas.SetLogx(1)
#canvas.SaveAs("exercise_0.png")

In [13]:
canvas.SaveAs(f"m_ks0_pi_{MODE}.pdf")

Info in <TCanvas::Print>: pdf file m_ks0_pi_LL.pdf has been created


## Helicity

In [12]:
rdf = rdf \
    .Define("vec_B", "ROOT::Math::PxPyPzEVector(B_PX, B_PY, B_PZ, B_ENERGY)") \
    .Define("vec_KS0", "ROOT::Math::PxPyPzEVector(KS0_PX, KS0_PY, KS0_PZ, KS0_ENERGY)") \
    .Define("vec_Pi", "ROOT::Math::PxPyPzEVector(Pi_Bachelor_PX, Pi_Bachelor_PY, Pi_Bachelor_PZ, Pi_Bachelor_ENERGY)") \
    .Define("vec_Kstar", "vec_KS0 + vec_Pi") \
    .Define("boost_Kstar", "vec_Kstar.BoostToCM()") \
    .Define("vec_B_in_Kstar", "ROOT::Math::VectorUtil::boost(vec_B, boost_Kstar)") \
    .Define("vec_KS0_in_Kstar", "ROOT::Math::VectorUtil::boost(vec_KS0, boost_Kstar)") \
    .Define("p3_B", "vec_B_in_Kstar.Vect()") \
    .Define("p3_KS0", "vec_KS0_in_Kstar.Vect()") \
    .Define("KSTAR_COS_THETA", "p3_B.Dot(p3_KS0) / (p3_B.R() * p3_KS0.R())")

In [13]:
particle = "KSTAR"
variable = "COS_THETA"

nbins = 100
limits = [-1,1]
unit = (limits[1]-limits[0])/nbins


can = ROOT.TCanvas()
can.SetLeftMargin(0.12)
hist_kstar = rdf.Filter("bdt_score > 0.9").Histo1D((f"{particle}_{variable}", f"{particle}_{variable}; {variable} [-]; Entries", nbins, limits[0], limits[1]), f"{particle}_{variable}")
hist_kstar.SetLineWidth(2)
# ROOT.gPad.SetLogy()
hist_kstar.Draw()
# ROOT.gPad.Update()  # forces ROOT to compute axis limits
# ymax = ROOT.gPad.GetUymax()   # top of y axis
# ymin = ROOT.gPad.GetUymin()   # bottom of y axis
can.Draw()

In [ ]:
particle = "B"
variable = "M_K0Pi"

limits = [750.0,1150.0]
nbins = 80
unit = (limits[1]-limits[0])/nbins
m_ks0_pi = ROOT.RooRealVar("m_ks0_pi","m_ks0_pi",limits[0],limits[1])

canvas = ROOT.TCanvas("canvas","canvas",800,600)
hist_kstar = rdf.Filter("(bdt_score > 0.9) & (abs(KSTAR_COS_THETA) > 0.3)").Histo1D((f"{particle}_{variable}", f"{particle}_{variable}; {variable} [MeV/c^2]; Entries", nbins, limits[0], limits[1]), f"{particle}_{variable}")
m_ks0_pi_hist = ROOT.RooDataHist("m_ks0_pi_hist","m_ks0_pi_hist",ROOT.RooArgList(m_ks0_pi),hist_kstar.GetValue())

#Create a drawable object and fill it
m_ks0_pi_plot = m_ks0_pi.frame(nbins) #Number of bins as an argument
m_ks0_pi_hist.plotOn(m_ks0_pi_plot, Name = "data")

#m_ks0_pi_plot.SetAxisRange(70.,751.)
m_ks0_pi_plot.SetTitle("")
ROOT.gPad.SetLeftMargin(0.15)
ROOT.gPad.SetBottomMargin(0.12)
m_ks0_pi_plot.GetYaxis().SetTitle("Candidates /(" + f'{unit:.2f}' + " MeV/#it{c}^{2})")
m_ks0_pi_plot.GetYaxis().SetTitleOffset(1.3)
m_ks0_pi_plot.GetXaxis().SetTitleOffset(1.0)
m_ks0_pi_plot.GetXaxis().SetTitleSize(0.05);
m_ks0_pi_plot.GetYaxis().SetTitleSize(0.05);
m_ks0_pi_plot.GetXaxis().SetTitle("#it{m}(#it{K_{S}^{0}#pi^{#pm})} [MeV/#it{c}^{2}]")
m_ks0_pi_plot.Draw()
canvas.Draw()
#canvas.SetLogx(1)
#canvas.SaveAs("exercise_0.png")

In [15]:
particle = "KSTAR"
variable = "COS_THETA"

nbins = 100
limits = [-1,1]
unit = (limits[1]-limits[0])/nbins


can = ROOT.TCanvas()
can.SetLeftMargin(0.12)
hist_kstar = rdf.Filter("(bdt_score > 0.9) & (B_M_K0Pi > 950)").Histo1D((f"{particle}_{variable}", f"{particle}_{variable}; {variable} [-]; Entries", nbins, limits[0], limits[1]), f"{particle}_{variable}")
hist_kstar.SetLineWidth(2)
# ROOT.gPad.SetLogy()
hist_kstar.Draw()
# ROOT.gPad.Update()  # forces ROOT to compute axis limits
# ymax = ROOT.gPad.GetUymax()   # top of y axis
# ymin = ROOT.gPad.GetUymin()   # bottom of y axis
can.Draw()

In [16]:
particle = "KSTAR"
variable = "COS_THETA"

nbins = 100
limits = [-1,1]
unit = (limits[1]-limits[0])/nbins


can = ROOT.TCanvas()
can.SetLeftMargin(0.12)
hist_kstar = rdf.Filter("(bdt_score > 0.9) & (B_M_K0Pi < 950) & (B_M_K0Pi > 820)").Histo1D((f"{particle}_{variable}", f"{particle}_{variable}; {variable} [-]; Entries", nbins, limits[0], limits[1]), f"{particle}_{variable}")
hist_kstar.SetLineWidth(2)
# ROOT.gPad.SetLogy()
hist_kstar.Draw()
# ROOT.gPad.Update()  # forces ROOT to compute axis limits
# ymax = ROOT.gPad.GetUymax()   # top of y axis
# ymin = ROOT.gPad.GetUymin()   # bottom of y axis
can.Draw()